# 0. Points of Interest

The aim of this notebook is to find, download, categorize and try and find potential patterns that may help identify areas of management and intervention

## Method overview
1. Download POIs from OSM, based on categories `"amenity", "shop", "leisure", "tourism"`
2. for each of the categories, we will be dropping all the points that have less than *10 counts*
3. Making a list of all subcategories under each tag and then try and make meaninful heuristic remapping. This is done using *Claude's Sonnet 5*. this step is more or less subjective and can be edited further, which is why this has been kept as a part of hard-coded text
4. saving the categorizations and plotting each of the categories

## Data provenance
OSM data was pulled on **13 August 2026**. OSM is a live, continuously-edited
dataset. Re-running this notebook at a later date may return a different network.

## File handling
Category summary is stored under `OUTPUTS_DIR`

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parents[1]
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

import geopandas as gpd
import pandas as pd
import contextily as cx
import matplotlib.pyplot as plt
import osmnx as ox
from shapely.geometry import box

from modules.config import (
    BBOX, 
    RAW_DATA, 
    OUTPUTS_DIR, 
    POI_BBOX
)

In [ ]:
result_gdf = gpd.read_file(os.path.join(OUTPUTS_DIR, "network_centrality.gpkg"))

In [ ]:
import geopandas as gpd
from shapely.geometry import LineString

# ── 1. Define Oxford Street as a reference line ──────────────────────────
# Rough coordinates along the corridor (Marble Arch -> Tottenham Court Rd),
# in WGS84 lon/lat. Doesn't need to be precise — it's just a spine to buffer.
oxford_ref = LineString([
    (-0.1590, 51.5136),  # Marble Arch end
    (-0.1310, 51.5165),  # Tottenham Court Rd / Centre Point end
])
oxford_ref_gdf = gpd.GeoSeries([oxford_ref], crs="EPSG:4326").to_crs(result_gdf.crs)

# ── 2. Buffer the reference line into a corridor ──────────────────────────
# 30m either side — covers carriageway + pavement + neatnet/QGIS snapping
# drift. Tightened up from an initial 15m, which missed segments near the
# pedestrianised section.
corridor = oxford_ref_gdf.buffer(30)

# ── 3. Capture segments falling inside the corridor ───────────────────────
oxford_mask = result_gdf.geometry.intersects(corridor.iloc[0])
oxford_street = result_gdf[oxford_mask]

# ── 4. Dissolve into a single line, then find what touches it ─────────────
# A tight buffer here (not the wide corridor) — this is testing genuine
# junction contact, not proximity. Wide would catch parallel streets that
# never actually meet Oxford Street.
oxford_dissolved = gpd.GeoSeries(
    [oxford_street.geometry.union_all()], crs=result_gdf.crs
)
touch_buffer = oxford_dissolved.buffer(100)

crossing = result_gdf[
    result_gdf.geometry.intersects(touch_buffer.iloc[0]) & ~oxford_mask
]

print(f"Oxford Street segments: {oxford_mask.sum()}")
print(f"Crossing segments: {len(crossing)}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8), dpi=300)

# Crossing streets drawn first, Oxford Street on top (zorder) so the
# corridor reads as one continuous line rather than being broken up
# visually by the streets crossing through it.
crossing.to_crs(epsg=3857).plot(
    ax=ax, color="steelblue", linewidth=1, zorder=1, label="Oxford Street area"
)
oxford_street.to_crs(epsg=3857).plot(
    ax=ax, color="red", linewidth=2.5, zorder=2, label="Oxford Street +\nintersecting arteries"
)

# contextily basemaps are Web Mercator (EPSG:3857) — reproject for
# plotting only, working CRS (EPSG:32630) stays untouched elsewhere.
cx.add_basemap(ax, source="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")

ax.set_axis_off()
ax.legend(loc="upper right", frameon=True)
ax.set_title("Oxford Street and intersecting arteries")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "oxford_street_crossings.png"), dpi=300)
plt.show()

In [ ]:
# ── 1. Define tag categories ───────────────────────────────────────────
# Start broad — amenity/shop/leisure covers most retail & hospitality use.
# tourism catches Oxford Street-relevant attractions (worth it given the
# corridor's mix of retail and visitor footfall).
POI_TAGS = {
    "amenity": True,
    "shop": True,
    "leisure": True,
    "tourism": True,
}

# ── 2. Pull from OSM, same extent as the network download ─────────────
# Use whatever polygon/bbox construction 01-intro.ipynb used for from_osm
# — reuse it directly rather than rebuilding, to guarantee identical extent.
pois_raw = ox.features_from_bbox(bbox=POI_BBOX, tags=POI_TAGS)

# ── 3. Persist raw pull before any processing ──────────────────────────
os.makedirs(RAW_DATA, exist_ok=True)
pois_raw_path = os.path.join(RAW_DATA, "pois_raw.gpkg")

# OSM features come back with mixed geometry types (Point, Polygon for
# buildings-as-amenities) and list-type columns GeoPackage can't store —
# drop/stringify non-scalar columns before writing.
pois_export = pois_raw.copy()
for col in pois_export.columns:
    if pois_export[col].apply(lambda x: isinstance(x, list)).any():
        pois_export[col] = pois_export[col].astype(str)

pois_export.to_file(pois_raw_path, driver="GPKG")

print(f"Pulled {len(pois_raw)} POIs")
print(pois_raw.geometry.geom_type.value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

# Oxford Street corridor for context, so you can see POI coverage/density
# relative to the street itself before the join happens

crossing.to_crs(epsg=3857).plot(
    ax=ax, color="steelblue", linewidth=1, zorder=1, label="Oxford Street area"
)
oxford_street.to_crs(epsg=3857).plot(
    ax=ax, color="red", linewidth=2.5, zorder=2, label="Oxford Street +\nintersecting arteries"
)

pois_export.to_crs(epsg=3857).plot(
    ax=ax, color="darkorange", markersize=8, alpha=0.6, zorder=1, label="POIs"
)

cx.add_basemap(ax, source="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")

# Build a geometry from POI_BBOX and reproject to match the plot's CRS (3857)
# osmnx bbox order varies by version — confirm yours below before trusting this
bbox_geom = gpd.GeoSeries([box(*POI_BBOX)], crs="EPSG:4326").to_crs(epsg=3857)
minx, miny, maxx, maxy = bbox_geom.total_bounds

ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

ax.set_axis_off()
ax.legend(loc="upper right", frameon=True)
ax.set_title(f"POIs pulled from OSM (n={len(pois_export)})")

plt.tight_layout()
plt.show()

In [ ]:
# Coalesce the four tag columns into one category label, prefixed by
# which OSM key it came from (a "restaurant" from amenity vs shop is
# meaningfully different, worth keeping distinguishable)
def consolidate_category(row):
    for key in ["amenity", "shop", "leisure", "tourism"]:
        val = row.get(key)
        if pd.notna(val):
            return f"{key}:{val}"
    return "other"

pois_export["category"] = pois_export.apply(consolidate_category, axis=1)

# Long tail of one-off tags makes for an unreadable legend — collapse
# anything outside the top N into "other" for plotting purposes
TOP_N = 12
top_categories = pois_export["category"].value_counts().head(TOP_N).index
pois_export["category_grouped"] = pois_export["category"].where(
    pois_export["category"].isin(top_categories), "other"
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

# Oxford Street corridor for context, so you can see POI coverage/density
# relative to the street itself before the join happens

crossing.to_crs(epsg=3857).plot(
    ax=ax, color="steelblue", linewidth=1, zorder=1, label="Oxford Street area"
)
oxford_street.to_crs(epsg=3857).plot(
    ax=ax, color="red", linewidth=2.5, zorder=2, label="Oxford Street +\nintersecting arteries"
)

pois_export.to_crs(epsg=3857).plot(
    ax=ax,
    column="category_grouped",
    categorical=True,
    legend=True,
    markersize=8,
    alpha=0.8,
    cmap="Set2",
    zorder=1,
    label="POIs",
    legend_kwds={"loc": "upper left", "bbox_to_anchor": (1.02, 1), "fontsize": 8},
)

cx.add_basemap(ax, source="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")

# Build a geometry from POI_BBOX and reproject to match the plot's CRS (3857)
# osmnx bbox order varies by version — confirm yours below before trusting this
bbox_geom = gpd.GeoSeries([box(*POI_BBOX)], crs="EPSG:4326").to_crs(epsg=3857)
minx, miny, maxx, maxy = bbox_geom.total_bounds

ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

ax.set_axis_off()
ax.legend(loc="upper right", frameon=True)
ax.set_title(f"POIs pulled from OSM (n={len(pois_export)})")

plt.tight_layout()
plt.show()

In [ ]:
# Consolidate category tags into one column, same approach as before —
# now applied to the point-only subset
def consolidate_category(row):
    for key in ["amenity", "shop", "leisure", "tourism"]:
        val = row.get(key)
        if pd.notna(val):
            return f"{key}:{val}"
    return "other"

pois_export["category"] = pois_export.apply(consolidate_category, axis=1)

TOP_N = 30
top_categories = pois_export["category"].value_counts().head(TOP_N).index
pois_export["category_grouped"] = pois_export["category"].where(
    pois_export["category"].isin(top_categories), "other"
)

print(pois_export["category_grouped"].value_counts())

In [ ]:
# ── Full tag-value breakdown, one row per category, all four keys ──────
category_counts = []
for key in ["amenity", "shop", "leisure", "tourism"]:
    if key in pois_export.columns:
        counts = pois_export[key].value_counts()
        for value, count in counts.items():
            category_counts.append({"osm_key": key, "osm_value": value, "count": count})

category_summary = pd.DataFrame(category_counts).sort_values(
    "count", ascending=False
).reset_index(drop=True)

print(f"Total distinct categories: {len(category_summary)}")
print(f"Total tagged points: {category_summary['count'].sum()} (of {len(pois_export)} — some points may lack all four tags, or carry more than one)")

os.makedirs(RAW_DATA, exist_ok=True)
category_summary.to_csv(os.path.join(RAW_DATA, "poi_category_summary.csv"), index=False)
category_summary.head(30)

In [ ]:
shop_summary = category_summary[category_summary["osm_key"] == "shop"].reset_index(drop=True)
print(f"Distinct shop types: {len(shop_summary)}")
print(f"Total shop-tagged points: {shop_summary['count'].sum()}")
shop_summary

In [ ]:
MIN_COUNT = 10

shop_keep = shop_summary[shop_summary["count"] >= MIN_COUNT]["osm_value"].tolist()
shop_drop = shop_summary[shop_summary["count"] < MIN_COUNT]

print(f"Keeping {len(shop_keep)} shop types (>= {MIN_COUNT} points)")
print(f"Dropping {len(shop_drop)} shop types, {shop_drop['count'].sum()} points total")

pois_shop = pois_export[pois_export["shop"].isin(shop_keep)].copy()
print(f"\npois_shop: {len(pois_shop)} points")
pois_shop["shop"].value_counts()

In [ ]:
shop_filtered = shop_summary[shop_summary["count"] >= 10].sort_values("count", ascending=False)

for _, row in shop_filtered.iterrows():
    print(f"{row['osm_value']:<25} {row['count']}")

## Shop category grouping

Raw OSM `shop=*` tags (84 distinct values, ≥10 points each) were grouped into
10 categories via manual thematic clustering rather than an existing retail
taxonomy, on the basis that OSM's tagging vocabulary does not map cleanly
onto standard retail classification schemes (e.g. SIC, Local Data Company
categories) at the point-tag level.

Grouping criteria: shared primary retail/service function (e.g. all apparel
items grouped under `fashion_apparel`; all garment/item repair services
grouped under `repair_services` as functionally distinct from retail-of-new-goods).

Two categories were deliberately kept ungrouped despite low individual counts,
due to direct relevance to the study's research question:
- `vacant` (n=245): vacancy rate is a variable of interest given the
  pedestrianisation intervention under study, not incidental noise.
- `department_store` (n=24): disproportionate footfall relevance relative to
  point count, given Oxford Street's anchor-store retail structure.

Ambiguous cases and their resolution:
- `optician` -> personal_care (health-adjacent service, not goods retail)
- `bookmaker` -> business_services (betting; not related to `books`)
- `charity` -> gifts_art_antiques (mixed/secondhand stock, no single theme)

Full mapping: see `SHOP_GROUPS` dict, [cell reference].

In [ ]:
SHOP_GROUPS = {
    # Fashion & Apparel
    "clothes": "fashion_apparel", "shoes": "fashion_apparel",
    "jewelry": "fashion_apparel", "watches": "fashion_apparel",
    "bag": "fashion_apparel", "fashion_accessories": "fashion_apparel",
    "leather": "fashion_apparel", "fabric": "fashion_apparel",

    # Food & Grocery Retail
    "convenience": "food_grocery", "supermarket": "food_grocery",
    "bakery": "food_grocery", "confectionery": "food_grocery",
    "deli": "food_grocery", "pastry": "food_grocery",
    "health_food": "food_grocery", "greengrocer": "food_grocery",
    "butcher": "food_grocery", "grocery": "food_grocery",
    "chocolate": "food_grocery", "tea": "food_grocery",
    "coffee": "food_grocery", "alcohol": "food_grocery",
    "wine": "food_grocery", "tobacco": "food_grocery",
    "e-cigarette": "food_grocery",

    # Personal Care & Grooming
    "hairdresser": "personal_care", "beauty": "personal_care",
    "cosmetics": "personal_care", "perfumery": "personal_care",
    "massage": "personal_care", "tattoo": "personal_care",
    "chemist": "personal_care", "optician": "personal_care",

    # Garment & Repair Services
    "dry_cleaning": "repair_services", "laundry": "repair_services",
    "tailor": "repair_services", "shoe_repair": "repair_services",
    "locksmith": "repair_services",

    # Home, Interior & DIY
    "furniture": "home_interior", "houseware": "home_interior",
    "hardware": "home_interior", "interior_decoration": "home_interior",
    "carpet": "home_interior", "kitchen": "home_interior",
    "bathroom_furnishing": "home_interior", "lighting": "home_interior",
    "flooring": "home_interior", "doityourself": "home_interior",
    "storage_rental": "home_interior",

    # Books, Media & Tech
    "books": "books_media_tech", "stationery": "books_media_tech",
    "newsagent": "books_media_tech", "photo": "books_media_tech",
    "camera": "books_media_tech", "music": "books_media_tech",
    "musical_instrument": "books_media_tech", "computer": "books_media_tech",
    "electronics": "books_media_tech", "mobile_phone": "books_media_tech",

    # Gifts, Art & Antiques
    "gift": "gifts_art_antiques", "art": "gifts_art_antiques",
    "antiques": "gifts_art_antiques", "frame": "gifts_art_antiques",
    "charity": "gifts_art_antiques", "toys": "gifts_art_antiques",
    "variety_store": "gifts_art_antiques",

    # Leisure, Sports & Vehicles
    "bicycle": "leisure_sports_vehicles", "sports": "leisure_sports_vehicles",
    "outdoor": "leisure_sports_vehicles", "car": "leisure_sports_vehicles",
    "car_repair": "leisure_sports_vehicles",

    # Professional & Business Services
    "travel_agency": "business_services", "estate_agent": "business_services",
    "copyshop": "business_services", "ticket": "business_services",
    "bookmaker": "business_services",

    # Vacant — kept separate deliberately, see note above
    "vacant": "vacant",

    # Department stores — separate flag, small count but high footfall relevance
    "department_store": "department_store",
}
# everything else (yes, mall, kiosk, pet, erotic) falls to "other" via fillna

pois_shop["shop_group"] = pois_shop["shop"].map(SHOP_GROUPS).fillna("other")

group_counts = pois_shop["shop_group"].value_counts()
print(group_counts)
print(f"\nTotal categorised: {group_counts.sum()} / {len(pois_shop)}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12), dpi=300)

pois_shop.to_crs(epsg=3857).plot(
    ax=ax,
    column="shop_group",
    categorical=True,
    legend=True,
    markersize=5,
    alpha=0.8,
    cmap="tab20",
    zorder=1,
    legend_kwds={"loc": "upper left", "bbox_to_anchor": (1.02, 1), "fontsize": 8},
)

crossing.to_crs(epsg=3857).plot(
    ax=ax, color="steelblue", linewidth=1, zorder=1, label="Oxford Street area"
)
oxford_street.to_crs(epsg=3857).plot(
    ax=ax, color="red", linewidth=2.5, zorder=2, label="Oxford Street +\nintersecting arteries"
)

cx.add_basemap(ax, source="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")

minx, miny, maxx, maxy = bbox_geom.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

ax.set_axis_off()
ax.set_title(f"Shop categories along Oxford Street and crossings (n={len(pois_shop)})")


plt.tight_layout()
plt.show()

In [ ]:
for key in ["amenity", "tourism", "leisure"]:
    key_summary = category_summary[category_summary["osm_key"] == key].sort_values(
        "count", ascending=False
    )
    key_filtered = key_summary[key_summary["count"] >= 10]

    print(f"\n{'='*50}")
    print(f"{key.upper()}  —  {len(key_filtered)} categories with >=10 points "
          f"(dropping {len(key_summary) - len(key_filtered)} smaller ones, "
          f"{key_summary[key_summary['count'] < 10]['count'].sum()} points)")
    print(f"{'='*50}")
    for _, row in key_filtered.iterrows():
        print(f"{row['osm_value']:<25} {row['count']}")

In [ ]:
AMENITY_GROUPS = {
    # Food & Drink — direct footfall/dwell-time destinations
    "restaurant": "food_drink", "cafe": "food_drink", "fast_food": "food_drink",
    "pub": "food_drink", "bar": "food_drink", "ice_cream": "food_drink",
    "food_court": "food_drink",

    # Nightlife, Arts & Culture
    "theatre": "nightlife_culture", "nightclub": "nightlife_culture",
    "cinema": "nightlife_culture", "casino": "nightlife_culture",
    "arts_centre": "nightlife_culture", "events_venue": "nightlife_culture",
    "studio": "nightlife_culture", "conference_centre": "nightlife_culture",
    "stripclub": "nightlife_culture",

    # Transport & Mobility
    "bicycle_parking": "transport_mobility", "motorcycle_parking": "transport_mobility",
    "parking": "transport_mobility", "parking_space": "transport_mobility",
    "parking_entrance": "transport_mobility", "bicycle_rental": "transport_mobility",
    "car_sharing": "transport_mobility", "taxi": "transport_mobility",
    "car_rental": "transport_mobility", "fuel": "transport_mobility",
    "ferry_terminal": "transport_mobility", "bicycle_repair_station": "transport_mobility",
    "charging_station": "transport_mobility", "ticket_validator": "transport_mobility",

    # Street Furniture & Infrastructure — NOT footfall destinations,
    # kept separate deliberately, see note below
    "bench": "street_infrastructure", "waste_basket": "street_infrastructure",
    "telephone": "street_infrastructure", "post_box": "street_infrastructure",
    "drinking_water": "street_infrastructure", "vending_machine": "street_infrastructure",
    "clock": "street_infrastructure", "shelter": "street_infrastructure",
    "grit_bin": "street_infrastructure", "waste_disposal": "street_infrastructure",
    "photo_booth": "street_infrastructure", "fountain": "street_infrastructure",

    # Financial Services
    "atm": "financial_services", "bank": "financial_services",
    "bureau_de_change": "financial_services",

    # Health Services
    "dentist": "health_services", "doctors": "health_services",
    "pharmacy": "health_services", "clinic": "health_services",
    "hospital": "health_services", "veterinary": "health_services",
    "social_facility": "health_services", "childcare": "health_services",

    # Education
    "school": "education", "university": "education", "college": "education",
    "kindergarten": "education", "language_school": "education",

    # Civic & Public Services
    "place_of_worship": "civic_public", "police": "civic_public",
    "courthouse": "civic_public", "post_office": "civic_public",
    "library": "civic_public", "community_centre": "civic_public",
    "toilets": "civic_public", "recycling": "civic_public",
    "marketplace": "civic_public",
}

pois_amenity = pois_export[pois_export["amenity"].notna()].copy()
pois_amenity["amenity_group"] = pois_amenity["amenity"].map(AMENITY_GROUPS).fillna("other")

group_counts = pois_amenity.groupby("amenity_group")["amenity"].agg(["count"])
print(group_counts.sort_values("count", ascending=False))
print(f"\n'other': {(pois_amenity['amenity_group'] == 'other').sum()} — should be ~0, all 67 filtered values are mapped above")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12), dpi=300)

pois_amenity.to_crs(epsg=3857).plot(
    ax=ax,
    column="amenity_group",
    categorical=True,
    legend=True,
    markersize=7,
    alpha=0.8,
    cmap="tab20",
    zorder=1,
    legend_kwds={"loc": "upper left", "bbox_to_anchor": (1.02, 1), "fontsize": 8},
)

crossing.to_crs(epsg=3857).plot(
    ax=ax, color="steelblue", linewidth=1, zorder=1, label="Oxford Street area"
)
oxford_street.to_crs(epsg=3857).plot(
    ax=ax, color="red", linewidth=2.5, zorder=2, label="Oxford Street +\nintersecting arteries"
)

cx.add_basemap(ax, source="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")

minx, miny, maxx, maxy = bbox_geom.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

ax.set_axis_off()
ax.set_title(f"Amenity categories along Oxford Street and crossings (n={len(pois_amenity)})")

plt.tight_layout()
plt.show()

In [ ]:
pois_amenity_relevant = pois_amenity[
    ~pois_amenity["amenity_group"].isin(["street_infrastructure", "transport_mobility"])
]

fig, ax = plt.subplots(figsize=(14, 12), dpi=300)

pois_amenity_relevant.to_crs(epsg=3857).plot(
    ax=ax, column="amenity_group", categorical=True, legend=True,
    markersize=7, alpha=0.8, cmap="tab20", zorder=1,
    legend_kwds={"loc": "upper left", "bbox_to_anchor": (1.02, 1), "fontsize": 8},
)
crossing.to_crs(epsg=3857).plot(ax=ax, color="steelblue", linewidth=1, zorder=2)
oxford_street.to_crs(epsg=3857).plot(ax=ax, color="red", linewidth=2, zorder=3)
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zoom="auto", alpha=0.7)
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.set_axis_off()
ax.set_title(f"Amenities (excl. infrastructure/parking) along Oxford Street (n={len(pois_amenity_relevant)})")

plt.tight_layout()
plt.show()

In [ ]:
TOURISM_GROUPS = {
    "hotel": "accommodation", "hostel": "accommodation",
    "guest_house": "accommodation", "apartment": "accommodation",

    "artwork": "culture_attraction", "gallery": "culture_attraction",
    "museum": "culture_attraction", "attraction": "culture_attraction",
    "viewpoint": "culture_attraction",

    "information": "visitor_services",
    # "yes" -> other via fillna
}

pois_tourism = pois_export[pois_export["tourism"].notna()].copy()
pois_tourism["tourism_group"] = pois_tourism["tourism"].map(TOURISM_GROUPS).fillna("other")
print(pois_tourism["tourism_group"].value_counts())

In [ ]:
LEISURE_GROUPS = {
    "garden": "green_space", "park": "green_space",
    "playground": "green_space", "picnic_table": "green_space",
    "bandstand": "green_space",

    "pitch": "sports_fitness", "fitness_centre": "sports_fitness",
    "sports_centre": "sports_fitness", "fitness_station": "sports_fitness",

    "outdoor_seating": "outdoor_seating",  # kept separate — hospitality
    # dwell-space adjacent to food_drink, not really "leisure" in the
    # green-space/sport sense, worth being able to isolate later
}

pois_leisure = pois_export[pois_export["leisure"].notna()].copy()
pois_leisure["leisure_group"] = pois_leisure["leisure"].map(LEISURE_GROUPS).fillna("other")
print(pois_leisure["leisure_group"].value_counts())

In [ ]:
from matplotlib.lines import Line2D
import matplotlib.cm as cm

fig, ax = plt.subplots(figsize=(14, 12), dpi=300)

# plot without auto-legend now — legend=False, we'll build it ourselves
leisure_cats = pois_leisure["leisure_group"].unique()
leisure_colors = {cat: cm.Set2(i) for i, cat in enumerate(sorted(leisure_cats))}
pois_leisure.to_crs(epsg=3857).plot(
    ax=ax, column="leisure_group", categorical=True, legend=False,
    markersize=18, alpha=0.8, cmap="Set2", zorder=1, marker="s",
)

tourism_cats = pois_tourism["tourism_group"].unique()
tourism_colors = {cat: cm.Set1(i) for i, cat in enumerate(sorted(tourism_cats))}
pois_tourism.to_crs(epsg=3857).plot(
    ax=ax, column="tourism_group", categorical=True, legend=False,
    markersize=18, alpha=0.8, cmap="Set1", zorder=1, marker="^",
)

crossing.to_crs(epsg=3857).plot(ax=ax, color="steelblue", linewidth=1, zorder=2)
oxford_street.to_crs(epsg=3857).plot(ax=ax, color="red", linewidth=2, zorder=3)
cx.add_basemap(ax, source="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")

ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.set_axis_off()
ax.set_title("Tourism (▲) and leisure (■) POIs along Oxford Street")

# manually built legend handles, marker shape matched to what was actually plotted
handles = (
    [Line2D([0], [0], marker="s", linestyle="", color=leisure_colors[c], label=c, markersize=8)
     for c in sorted(leisure_cats)]
    + [Line2D([0], [0], marker="^", linestyle="", color=tourism_colors[c], label=c, markersize=8)
       for c in sorted(tourism_cats)]
)
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)

plt.tight_layout()
plt.show()